# Library

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import pickle

In [2]:
BASE_PATH = Path(".")
NB2_PATH  = BASE_PATH / "nb2"
NB3_PATH  = BASE_PATH / "nb3"
NB3_PATH.mkdir(exist_ok=True, parents=True)

# Inputs from NB1
HISN_PARQUET   = NB2_PATH / "hisn_final.parquet"
HISN_FEAT_NPY  = NB2_PATH / "hisn_review_features.npy"
USER_STATS_CSV = NB2_PATH / "user_stats.csv"
ITEM_STATS_CSV = NB2_PATH / "item_stats.csv"

# Load HISN Reviews + Features and Sanity Check

In this code block 

- Load the HISN review subset (hisn_final.parquet)
- Load the HISN review feature matrix (hisn_review_features.npy)
- Reset index to have clean 0..N-1 indexing
- Assert that the number of rows in hisn equals the number of feature rows → guarantees alignment between rows and feature vectors

This alignment is critical: SL-GAD will assume row i of the features corresponds to row i of the HISN review table.

In [3]:
hisn = pd.read_parquet(HISN_PARQUET).reset_index(drop=True)
hisn_feats = np.load(HISN_FEAT_NPY)

print("HISN shape:", hisn.shape)
print("HISN features shape:", hisn_feats.shape)

assert hisn.shape[0] == hisn_feats.shape[0], (
    f"HISN rows ({hisn.shape[0]}) != hisn_review_features rows ({hisn_feats.shape[0]})"
)

HISN shape: (7378, 31)
HISN features shape: (7378, 777)


In [23]:
hisn.head(3).T

,0,1,2
rating,5,5,3
title,Fantastic,Very nice,Nice shirt poor sizing
text,What a Beauty. The material is canvas and very...,"So pretty , delicate. Wish the chain was a bit...",The cats are beautiful but for the size (x-lar...
asin,B07D5TV7VS,B01N8P8Q9Q,B06Y26ZMBG
parent_asin,B07D5TV7VS,B01N8P8Q9Q,B06Y26ZMBG
user_id,AHTWISCZKNLEBVTIFZP6EZRY2ISA,AHTWISCZKNLEBVTIFZP6EZRY2ISA,AHTWISCZKNLEBVTIFZP6EZRY2ISA
timestamp,2018-10-15 01:37:25.222000,2018-04-13 06:55:56.008000,2018-02-05 04:42:57.931000
helpful_vote,0,2,1
verified_purchase,True,True,True
full_text,fantastic what a beauty. the material is canva...,"very nice so pretty , delicate. wish the chain...",nice shirt poor sizing the cats are beautiful ...


# Assign Review IDs + Build ID Lists & Mappings

In this code block we do
- Add a stable review identifier review_id of the form r_<index> → used as node_id for review nodes
- Save this enriched HISN parquet for other notebooks if needed
- Extract unique IDs of users, products, and reviews
- Build specific id mapping
- Save the mapping for next notebook.

In [4]:
hisn["review_id"] = [f"r_{i}" for i in range(len(hisn))]
hisn.to_parquet(NB2_PATH / "hisn_final_with_rid.parquet", index=False)

# Unique node IDs per type (as strings)
user_ids   = hisn["user_id"].astype(str).unique().tolist()
prod_ids   = hisn["asin"].astype(str).unique().tolist()
review_ids = hisn["review_id"].astype(str).tolist()

print("Num users:", len(user_ids))
print("Num products:", len(prod_ids))
print("Num reviews:", len(review_ids))

# Local index per node type (0..n_type-1)
usr2idx = {u: i for i, u in enumerate(user_ids)}
prd2idx = {a: i for i, a in enumerate(prod_ids)}
rev2idx = {r: i for i, r in enumerate(review_ids)}

with open(NB3_PATH / "hisn_mappings.pkl", "wb") as f:
    pickle.dump(
        {"user2idx": usr2idx, "item2idx": prd2idx, "review2idx": rev2idx},
        f
    )
print("✅ Saved hisn_mappings.pkl")

Num users: 2755
Num products: 2976
Num reviews: 7378
✅ Saved hisn_mappings.pkl


# Load User & Item Stats, Align to HISN IDs

In this block we gonna load the global user statistics and the item from NB1 and filter them down only with the ones that appear in HISN. This is crucial to make sure the alignment of IDs

In [5]:
user_stats = pd.read_csv(USER_STATS_CSV)
item_stats = pd.read_csv(ITEM_STATS_CSV)

user_stats = user_stats[user_stats["user_id"].isin(user_ids)].copy()
item_stats = item_stats[item_stats["asin"].isin(prod_ids)].copy()

In [6]:
user_stats = (
    user_stats
    .set_index("user_id")
    .reindex(user_ids)   # follow user_ids order
    .reset_index()
)
item_stats = (
    item_stats
    .set_index("asin")
    .reindex(prod_ids)   # follow prod_ids order
    .reset_index()
)

In [7]:
user_stats = user_stats.fillna(0)
item_stats = item_stats.fillna(0)

print("Aligned user_stats shape:", user_stats.shape)
print("Aligned item_stats shape:", item_stats.shape)


Aligned user_stats shape: (2755, 10)
Aligned item_stats shape: (2976, 6)


In [19]:
user_stats.head()

,user_id,n_reviews_user,avg_rating_user,std_rating_user,frac_verified_user,avg_len_user,dup_ratio_user,max_reviews_per_day,day_entropy,avg_text_sim
0,AHTWISCZKNLEBVTIFZP6EZRY2ISA,3,4.333333,1.154701,1.0,52.666667,0.0,1,1.098612e+00,0.422592
1,AF45GMG7WO3TWKU34ADJGD3IMSHA,2,5.000000,0.000000,1.0,13.000000,1.0,2,-1.000000e-10,1.000000
2,AEZN2MJLB3VOZFNUT4RG2DOWFC4A,2,5.000000,0.000000,1.0,9.000000,1.0,2,-1.000000e-10,1.000000
3,AH3QNNM6B5FDM4DZZH4KFUXP74XQ,4,2.500000,0.577350,1.0,52.500000,1.0,4,-1.000000e-10,0.692410
4,AEK6XGWJ3SGTNIGWZ2SAVJFJ7BZA,4,5.000000,0.000000,1.0,12.000000,1.0,2,6.931472e-01,0.453181


In [20]:
item_stats.head()

,asin,n_reviews_item,avg_rating_item,std_rating_item,dup_ratio_item,n_unique_users
0,B07D5TV7VS,3,5.000000,0.000000,0.666667,2
1,B01N8P8Q9Q,3,5.000000,0.000000,0.666667,2
2,B06Y26ZMBG,12,3.750000,1.356801,0.000000,12
3,B00J3TQ3ZU,6,3.666667,2.065591,0.666667,4
4,B075Q7HZ9M,3,5.000000,0.000000,0.666667,2


# Build Node Tables (user, product, review)

Construct user node table:

- node_id = original user_id (string)
- node_type = "user"
- All other columns prefixed with user_ to avoid name collisions

Construct product node table similarly:
- node_id = asin
- node_type = "product"
- Prefix features with item_

Construct review node table:
- node_id = review_id
- node_type = "review"

These tables will be used to write nodes.csv and to extract type-specific feature matrices.

In [9]:
# --- User nodes ---
user_nodes = user_stats.rename(columns={"user_id": "node_id"})
user_nodes["node_id"] = user_nodes["node_id"].astype(str)
user_nodes["node_type"] = "user"

# Put ID + type first, then features
user_nodes = user_nodes[["node_id", "node_type"] +
                        [c for c in user_nodes.columns if c not in ["node_id", "node_type"]]]

# Prefix feature columns with "user_"
user_nodes = user_nodes.rename(
    columns={c: f"user_{c}" for c in user_nodes.columns if c not in ["node_id", "node_type"]}
)

In [16]:
user_nodes.head()

,node_id,node_type,user_n_reviews_user,user_avg_rating_user,user_std_rating_user,user_frac_verified_user,user_avg_len_user,user_dup_ratio_user,user_max_reviews_per_day,user_day_entropy,user_avg_text_sim
0,AHTWISCZKNLEBVTIFZP6EZRY2ISA,user,3,4.333333,1.154701,1.0,52.666667,0.0,1,1.098612e+00,0.422592
1,AF45GMG7WO3TWKU34ADJGD3IMSHA,user,2,5.000000,0.000000,1.0,13.000000,1.0,2,-1.000000e-10,1.000000
2,AEZN2MJLB3VOZFNUT4RG2DOWFC4A,user,2,5.000000,0.000000,1.0,9.000000,1.0,2,-1.000000e-10,1.000000
3,AH3QNNM6B5FDM4DZZH4KFUXP74XQ,user,4,2.500000,0.577350,1.0,52.500000,1.0,4,-1.000000e-10,0.692410
4,AEK6XGWJ3SGTNIGWZ2SAVJFJ7BZA,user,4,5.000000,0.000000,1.0,12.000000,1.0,2,6.931472e-01,0.453181


In [10]:
# --- Product (item) nodes ---
item_nodes = item_stats.rename(columns={"asin": "node_id"})
item_nodes["node_id"] = item_nodes["node_id"].astype(str)
item_nodes["node_type"] = "product"

item_nodes = item_nodes[["node_id", "node_type"] +
                        [c for c in item_nodes.columns if c not in ["node_id", "node_type"]]]

item_nodes = item_nodes.rename(
    columns={c: f"item_{c}" for c in item_nodes.columns if c not in ["node_id", "node_type"]}
)

In [15]:
item_nodes.head()

,node_id,node_type,item_n_reviews_item,item_avg_rating_item,item_std_rating_item,item_dup_ratio_item,item_n_unique_users
0,B07D5TV7VS,product,3,5.000000,0.000000,0.666667,2
1,B01N8P8Q9Q,product,3,5.000000,0.000000,0.666667,2
2,B06Y26ZMBG,product,12,3.750000,1.356801,0.000000,12
3,B00J3TQ3ZU,product,6,3.666667,2.065591,0.666667,4
4,B075Q7HZ9M,product,3,5.000000,0.000000,0.666667,2


In [11]:
# --- Review nodes ---
review_nodes = pd.DataFrame({
    "node_id": review_ids,
    "node_type": "review"
})

In [17]:
review_nodes.head()

,node_id,node_type
0,r_0,review
1,r_1,review
2,r_2,review
3,r_3,review
4,r_4,review


# Extract and Save Node Feature Matrices

We now:

- Extract user feature matrix from user_nodes
- Extract item feature matrix from item_nodes
- Store them as float32 .npy arrays
- also save review feature matrix using hisn_feats from NB1 as review_node_features.npy.

We ensure these arrays are aligned:
- Row i in user_node_features.npy corresponds to user_ids[i]
- Row i in item_node_features.npy corresponds to prod_ids[i]
- Row i in review_node_features.npy corresponds to review_ids[i] (i.e., hisn row i)

In [12]:
# User features
user_feat_cols = [c for c in user_nodes.columns if c not in ["node_id", "node_type"]]
user_feat_matrix = user_nodes[user_feat_cols].values.astype(np.float32)
np.save(NB3_PATH / "user_node_features.npy", user_feat_matrix)

# Item (product) features
item_feat_cols = [c for c in item_nodes.columns if c not in ["node_id", "node_type"]]
item_feat_matrix = item_nodes[item_feat_cols].values.astype(np.float32)
np.save(NB3_PATH / "item_node_features.npy", item_feat_matrix)

# Review features (from HISN feature matrix)
review_feat_matrix = hisn_feats.astype(np.float32)
np.save(NB3_PATH / "review_node_features.npy", review_feat_matrix)

In [13]:
print(f"Saved user features:   {user_feat_matrix.shape}")
print(f"Saved item features:   {item_feat_matrix.shape}")
print(f"Saved review features: {review_feat_matrix.shape}")

Saved user features:   (2755, 9)
Saved item features:   (2976, 5)
Saved review features: (7378, 777)


# Build Edge List

We build four directed edge types:

- user -> review : "user-writes-review"
- review -> item : "review-about-item"
- review -> user : "review-written-by-user" (reverse)
- item -> review : "item-has-review" (reverse)

We:
- Keep src and dst as string IDs for now (NB3 will convert them to integer indices using usr2idx, prd2idx, rev2idx).
- Save to nb3/edges.csv

In [24]:
edges_ur = pd.DataFrame({
    "src": hisn["user_id"].astype(str),
    "dst": hisn["review_id"].astype(str),
    "edge_type": "user-writes-review"      # user --> review
})

edges_ri = pd.DataFrame({
    "src": hisn["review_id"].astype(str),
    "dst": hisn["asin"].astype(str),
    "edge_type": "review-about-item"       # review --> product
})

edges_ru = pd.DataFrame({
    "src": hisn["review_id"].astype(str),
    "dst": hisn["user_id"].astype(str),
    "edge_type": "review-written-by-user"  # review --> user (reverse)
})

edges_ir = pd.DataFrame({
    "src": hisn["asin"].astype(str),
    "dst": hisn["review_id"].astype(str),
    "edge_type": "item-has-review"         # product --> review (reverse)
})

In [25]:
edges = pd.concat([edges_ur, edges_ri, edges_ru, edges_ir], ignore_index=True)
edges.to_csv(NB3_PATH / "edges.csv", index=False)

In [27]:
edges['edge_type'].value_counts()

edge_type
user-writes-review        7378
review-about-item         7378
review-written-by-user    7378
item-has-review           7378
Name: count, dtype: int64

In [29]:
pd.concat([edges.head(), edges.tail()])

,src,dst,edge_type
0,AHTWISCZKNLEBVTIFZP6EZRY2ISA,r_0,user-writes-review
1,AHTWISCZKNLEBVTIFZP6EZRY2ISA,r_1,user-writes-review
2,AHTWISCZKNLEBVTIFZP6EZRY2ISA,r_2,user-writes-review
3,AF45GMG7WO3TWKU34ADJGD3IMSHA,r_3,user-writes-review
4,AF45GMG7WO3TWKU34ADJGD3IMSHA,r_4,user-writes-review
29507,B00CJUHGR4,r_7373,item-has-review
29508,B019MO9B0O,r_7374,item-has-review
29509,B019MO9B0O,r_7375,item-has-review
29510,B08CZN5TR4,r_7376,item-has-review
29511,B08CZN5TR4,r_7377,item-has-review


# Save Nodes CSV

Finally we:

- Ensure review_nodes has correct dtype (string IDs)
- Concatenate user_nodes, item_nodes, review_nodes into a single nodes DataFrame
- Save as nodes.csv for debugging / visual inspection / NB3 convenience
- Print node counts per type

This CSV is not strictly necessary for model training, but is great for sanity checks and for building visualization tools.

In [30]:
review_nodes = review_nodes.copy()
review_nodes["node_id"] = review_nodes["node_id"].astype(str)
review_nodes["node_type"] = "review"

nodes = pd.concat([user_nodes, item_nodes, review_nodes], ignore_index=True)
nodes.to_csv(NB3_PATH / "nodes.csv", index=False)

In [32]:
nodes.head(3).T

,0,1,2
node_id,AHTWISCZKNLEBVTIFZP6EZRY2ISA,AF45GMG7WO3TWKU34ADJGD3IMSHA,AEZN2MJLB3VOZFNUT4RG2DOWFC4A
node_type,user,user,user
user_n_reviews_user,3.0,2.0,2.0
user_avg_rating_user,4.333333,5.0,5.0
user_std_rating_user,1.154701,0.0,0.0
user_frac_verified_user,1.0,1.0,1.0
user_avg_len_user,52.666667,13.0,9.0
user_dup_ratio_user,0.0,1.0,1.0
user_max_reviews_per_day,1.0,2.0,2.0
user_day_entropy,1.098612,-0.0,-0.0


In [33]:
nodes["node_type"].value_counts()

node_type
review     7378
product    2976
user       2755
Name: count, dtype: int64